# Fine-Tuning Hyperparameters and Model Size


In [ ]:
from pathlib import Path
import re
import sys
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns


In [ ]:
repo_dir = Path("../../..")

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

from visualization.src.utils import save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, ARCHITECUTURE_FAMILY_COLORS


In [ ]:
sns.set_theme(style="whitegrid")
sns.set_theme(style="ticks")

artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"


In [ ]:
results_file = artifacts_dir / "finetuning_results.csv"
df_all = pd.read_csv(results_file, low_memory=False)

df_all.shape


In [ ]:
HEATMAP_VAL_MULTIPLIER = 100

METRICS = ["pearsonr_nc", "rsa_c_test", "cka_c_test"]
AGG_COLS = METRICS + ["task_evaluation_acc"]

BENCHMARKS = [
    "bs_fz",
    "bs_mh",
    "tvsd",
    "things_fmri",
    "nsd_func1pt8mm_individualROIs",
    "things_eeg1",
    "things_eeg2",
    "things_meg",
]
BENCHMARKS_WITH_AVERAGE = BENCHMARKS + ["benchmark_average"]

FINETUNING_DATASETS = [
    "tvsd",
    "things_fmri",
    "nsd_func1pt8mm_individualROIs",
    "things_eeg1",
    "things_eeg2",
    "things_meg",
]

MODEL_LABELS = {
    "deit_small": "ViT-S",
    "deit_base": "ViT-B",
    "deit_large": "ViT-L",
}
MODEL_ORDER = ["ViT-S", "ViT-B", "ViT-L"]
MODEL_PALETTE = {
    "ViT-S": mcolors.to_hex(mcolors.to_rgb(ARCHITECUTURE_FAMILY_COLORS["ViT"])),
    "ViT-B": "#2A9D8F",
    "ViT-L": "#457B9D",
}

benchmark_order = {benchmark: idx for idx, benchmark in enumerate(BENCHMARKS_WITH_AVERAGE)}
finetuning_dataset_order = {dataset: idx for idx, dataset in enumerate(FINETUNING_DATASETS)}


In [ ]:
def parse_lr_label(model_id: str) -> str:
    # Convert model id tokens such as lr1e5_0 to the paper-readable label 1e-5.
    match = re.search(r"-lr(?P<mantissa>\d+)e(?P<exp>\d+)(?:_\d+)?(?:-|$)", model_id)
    if match is None:
        return np.nan
    return f"{match.group('mantissa')}e-{match.group('exp')}"


def lr_label_to_float(lr_label: str) -> float:
    mantissa, exponent = lr_label.split("e-")
    return float(mantissa) * 10 ** (-int(exponent))


def get_base_model_id(model_id: str) -> str:
    arch_map = {
        "deit_small": "deit_small_imagenet_full_seed-0",
        "deit_base": "deit_base_imagenet_full_seed-0",
        "deit_large": "deit_large_imagenet_full_seed-0",
    }
    return arch_map[model_id.split("-")[0]]


def get_model_name(model_id: str) -> str:
    return MODEL_LABELS[model_id.split("-")[0]]


def add_pretty_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["benchmark_label"] = df["benchmark_name"].map(BENCHMARK_NAME_MAPPING)
    df["finetuning_dataset_label"] = df["finetuning_dataset"].map(BENCHMARK_NAME_MAPPING)
    df["benchmark_order"] = df["benchmark_name"].map(benchmark_order)
    df["finetuning_dataset_order"] = df["finetuning_dataset"].map(finetuning_dataset_order)
    return df


In [ ]:
def build_explore_diffs(df_all: pd.DataFrame) -> pd.DataFrame:
    df_finetuned = df_all[
        (df_all["exp_type"] == "finetuning_explore")
        & (df_all["data_pct"] == 100)
        & (df_all["arch"].isin(MODEL_LABELS))
    ].copy()
    df_finetuned["base_model_id"] = df_finetuned["model_id"].apply(get_base_model_id)
    df_finetuned["model_name"] = df_finetuned["model_id"].apply(get_model_name)
    df_finetuned["lr_label"] = df_finetuned["model_id"].apply(parse_lr_label)
    df_finetuned["epochs"] = df_finetuned["epochs"].astype(int)
    df_finetuned["seed"] = df_finetuned["seed"].astype(int)
    df_finetuned["hyperparam"] = df_finetuned.apply(
        lambda row: f"ep{row['epochs']}, lr {row['lr_label']}", axis=1
    )

    df_finetuned = apply_hiearchical_aggregation(
        df=df_finetuned,
        groupby_cols=[
            "model_id",
            "benchmark_name",
            "seed",
            "finetuning_dataset",
            "arch",
            "epochs",
            "lr_label",
            "hyperparam",
            "base_model_id",
            "model_name",
        ],
        agg_cols=AGG_COLS,
        agg_func="mean",
    )

    baseline_ids = list({get_base_model_id(model) for model in MODEL_LABELS})
    df_baseline = df_all[
        (df_all["exp_type"] == "finetuning_baseline")
        & (df_all["model_id"].isin(baseline_ids))
    ].copy()
    df_baseline = apply_hiearchical_aggregation(
        df=df_baseline,
        groupby_cols=["model_id", "benchmark_name"],
        agg_cols=AGG_COLS,
        agg_func="mean",
    )

    df_diff = df_finetuned.merge(
        df_baseline,
        left_on=["base_model_id", "benchmark_name"],
        right_on=["model_id", "benchmark_name"],
        suffixes=("_finetuned", "_baseline"),
    )

    for metric in AGG_COLS:
        df_diff[f"{metric}_diff"] = df_diff[f"{metric}_finetuned"] - df_diff[f"{metric}_baseline"]

    avg_group_cols = [
        "model_id_finetuned",
        "base_model_id",
        "seed",
        "finetuning_dataset",
        "arch",
        "epochs",
        "lr_label",
        "hyperparam",
        "model_name",
    ]
    value_cols = [f"{metric}_diff" for metric in AGG_COLS]
    df_avg = df_diff.groupby(avg_group_cols, dropna=False)[value_cols].mean().reset_index()
    df_avg["benchmark_name"] = "benchmark_average"

    keep_cols = avg_group_cols + ["benchmark_name"] + value_cols
    df_diff = pd.concat([df_diff[keep_cols], df_avg], ignore_index=True)
    return add_pretty_labels(df_diff)


df_diff = build_explore_diffs(df_all)
df_diff.shape


In [ ]:
df_diff[[
    "model_name",
    "finetuning_dataset",
    "benchmark_name",
    "epochs",
    "lr_label",
    "seed",
    "pearsonr_nc_diff",
]].head()


## Model Size


In [ ]:
BASE_EPOCHS = 20
BASE_LR_LABEL = "1e-5"
USE_SHARED_SEED_FOR_MODEL_SIZE = False

model_size_df = df_diff[
    (df_diff["epochs"] == BASE_EPOCHS)
    & (df_diff["lr_label"] == BASE_LR_LABEL)
    & (df_diff["model_name"].isin(MODEL_ORDER))
].copy()

if USE_SHARED_SEED_FOR_MODEL_SIZE:
    model_size_df = model_size_df[model_size_df["seed"] == 0].copy()

model_size_df["model_name"] = pd.Categorical(model_size_df["model_name"], MODEL_ORDER, ordered=True)
model_size_df["benchmark_name"] = pd.Categorical(
    model_size_df["benchmark_name"], BENCHMARKS_WITH_AVERAGE, ordered=True
)
model_size_df = model_size_df.sort_values(
    ["benchmark_order", "finetuning_dataset_order", "model_name"]
)

model_size_summary = model_size_df.groupby(
    ["benchmark_name", "benchmark_label", "benchmark_order", "model_name"],
    observed=True,
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
    pearsonr_nc_sem=("pearsonr_nc_diff", "sem"),
    n_finetuning_datasets=("finetuning_dataset", "nunique"),
).reset_index()
model_size_summary["pearsonr_nc_sem"] = model_size_summary["pearsonr_nc_sem"].fillna(0)
model_size_summary = model_size_summary.sort_values(["benchmark_order", "model_name"])

model_size_summary.pivot_table(
    index="model_name",
    columns="benchmark_label",
    values="pearsonr_nc_mean",
    aggfunc="mean",
)


In [ ]:
model_size_detail_summary = model_size_df.groupby(
    [
        "benchmark_name",
        "benchmark_label",
        "benchmark_order",
        "finetuning_dataset_label",
        "finetuning_dataset_order",
        "model_name",
    ],
    observed=True,
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
    pearsonr_nc_sem=("pearsonr_nc_diff", "sem"),
).reset_index()
model_size_detail_summary["pearsonr_nc_sem"] = model_size_detail_summary["pearsonr_nc_sem"].fillna(0)
model_size_detail_summary["n_seeds"] = model_size_df["seed"].nunique()

model_size_detail_average = model_size_df.groupby(
    ["benchmark_name", "benchmark_label", "benchmark_order", "model_name"],
    observed=True,
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
    pearsonr_nc_sem=("pearsonr_nc_diff", "sem"),
    n_finetuning_datasets=("finetuning_dataset", "nunique"),
).reset_index()
model_size_detail_average["pearsonr_nc_sem"] = model_size_detail_average["pearsonr_nc_sem"].fillna(0)
model_size_detail_average["finetuning_dataset_label"] = "Average"
model_size_detail_average["finetuning_dataset_order"] = len(FINETUNING_DATASETS)

model_size_detail_summary = pd.concat(
    [model_size_detail_summary, model_size_detail_average],
    ignore_index=True,
)

finetuning_dataset_labels_with_average = [
    BENCHMARK_NAME_MAPPING[dataset] for dataset in FINETUNING_DATASETS
] + ["Average"]

fig, axes = plt.subplots(3, 3, figsize=(15.5, 10.5), dpi=300, sharey=True)
axes = axes.flatten()
bar_width = 0.24
x = np.arange(len(finetuning_dataset_labels_with_average))
offsets = np.linspace(-bar_width, bar_width, len(MODEL_ORDER))

for idx, (ax, benchmark) in enumerate(zip(axes, BENCHMARKS_WITH_AVERAGE)):
    data = model_size_detail_summary[
        model_size_detail_summary["benchmark_name"] == benchmark
    ].copy()
    data = data.set_index(["finetuning_dataset_label", "model_name"])

    for offset, model_name in zip(offsets, MODEL_ORDER):
        heights = data.loc[
            [(label, model_name) for label in finetuning_dataset_labels_with_average],
            "pearsonr_nc_mean",
        ].values
        errors = data.loc[
            [(label, model_name) for label in finetuning_dataset_labels_with_average],
            "pearsonr_nc_sem",
        ].values
        ax.bar(
            x + offset,
            heights,
            width=bar_width,
            yerr=errors,
            label=model_name,
            color=MODEL_PALETTE[model_name],
            edgecolor="0.2",
            linewidth=0.5,
            capsize=2,
            error_kw={"elinewidth": 0.8, "capthick": 0.8, "ecolor": "0.2"},
        )

    ax.axhline(0, color="0.2", linewidth=0.8)
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark], fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(r"$\Delta$ Alignment" if idx % 3 == 0 else "", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(finetuning_dataset_labels_with_average, rotation=45, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.spines[["top", "right"]].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles[:len(MODEL_ORDER)], labels[:len(MODEL_ORDER)], title=None, loc="center", ncol=len(MODEL_ORDER), frameon=False, bbox_to_anchor=(0.5, 0.98))
fig.suptitle("ViT Model Size by Fine-Tuning Dataset", fontsize=18, fontweight="bold", y=1.03)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="finetuning-hyperopt-model_size-by_benchmark_dataset",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Model Size by Fine-Tuning Dataset**

\textbf{finetuning-hyperopt-model_size-by_benchmark_dataset:} Change in neural alignment after fine-tuning ViT-S, ViT-B, and ViT-L for $20$ epochs at learning rate $10^{-5}$, split by fine-tuning dataset and evaluation benchmark. Per-dataset bars: SEM across 3 seeds; Average bars: SEM across 6 datasets $\times$ 3 seeds.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(9, 8.5), dpi=300, sharey=True)
axes = axes.flatten()

for ax, benchmark in zip(axes, BENCHMARKS_WITH_AVERAGE):
    data = model_size_summary[model_size_summary["benchmark_name"] == benchmark].copy()
    x = np.arange(len(MODEL_ORDER))
    heights = data.set_index("model_name").loc[MODEL_ORDER, "pearsonr_nc_mean"].values
    errors = data.set_index("model_name").loc[MODEL_ORDER, "pearsonr_nc_sem"].values
    colors = [MODEL_PALETTE[model_name] for model_name in MODEL_ORDER]

    ax.bar(
        x,
        heights,
        yerr=errors,
        color=colors,
        edgecolor="0.2",
        linewidth=0.6,
        capsize=3,
        error_kw={"elinewidth": 1.0, "capthick": 1.0, "ecolor": "0.2"},
    )
    ax.axhline(0, color="0.2", linewidth=0.8)
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark], fontsize=13, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(r"$\Delta$ Alignment" if benchmark in ["bs_fz", "things_fmri", "things_eeg2"] else "", fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_ORDER, rotation=0)
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.spines[["top", "right"]].set_visible(False)

handles = [plt.Rectangle((0, 0), 1, 1, color=MODEL_PALETTE[model]) for model in MODEL_ORDER]
# fig.legend(handles, MODEL_ORDER, loc="center", ncol=len(MODEL_ORDER), frameon=False, bbox_to_anchor=(0.5, 0.98))
fig.suptitle("ViT Model Size", fontsize=18, fontweight="bold")
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="finetuning-hyperopt-model_size-average_by_benchmark",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Model Size Averaged Across Fine-Tuning Datasets**

\textbf{finetuning-hyperopt-model_size-average_by_benchmark:} Change in neural alignment, $\Delta r_{\mathrm{NC}}$, for ViT model sizes at the shared training setting, averaged across 6 fine-tuning datasets and 3 seeds. Error bars: SEM across 6 datasets $\times$ 3 seeds.


In [ ]:
model_size_by_finetuning_dataset = model_size_df[
    model_size_df["benchmark_name"] == "benchmark_average"
].groupby(
    ["finetuning_dataset_label", "model_name"], observed=True
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
).reset_index()

model_size_by_finetuning_dataset = model_size_by_finetuning_dataset.pivot_table(
    index="finetuning_dataset_label",
    columns="model_name",
    values="pearsonr_nc_mean",
    aggfunc="mean",
).loc[
    [BENCHMARK_NAME_MAPPING[dataset] for dataset in FINETUNING_DATASETS],
    MODEL_ORDER,
]
model_size_by_finetuning_dataset.round(5)


In [ ]:
selected_benchmark_model_size = model_size_df[
    model_size_df["benchmark_name"].isin(["things_fmri", "things_meg", "things_eeg2", "bs_mh"])
].groupby(
    ["benchmark_label", "finetuning_dataset_label", "model_name"], observed=True
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
).reset_index()

selected_benchmark_model_size = selected_benchmark_model_size.pivot_table(
    index=["benchmark_label", "finetuning_dataset_label"],
    columns="model_name",
    values="pearsonr_nc_mean",
    aggfunc="mean",
).loc[
    pd.MultiIndex.from_product(
        [
            [BENCHMARK_NAME_MAPPING[benchmark] for benchmark in ["things_fmri", "things_meg", "things_eeg2", "bs_mh"]],
            [BENCHMARK_NAME_MAPPING[dataset] for dataset in FINETUNING_DATASETS],
        ],
        names=["benchmark_label", "finetuning_dataset_label"],
    ),
    MODEL_ORDER,
]
selected_benchmark_model_size.round(5)


## Hyperparameters


In [ ]:
hyperparam_df = df_diff[
    (df_diff["model_name"] == "ViT-S")
    & (df_diff["benchmark_name"] == "benchmark_average")
].copy()

hyperparam_summary = hyperparam_df.groupby(
    ["epochs", "lr_label", "hyperparam"], dropna=False
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
    pearsonr_nc_sem=("pearsonr_nc_diff", "sem"),
    rsa_c_test_mean=("rsa_c_test_diff", "mean"),
    cka_c_test_mean=("cka_c_test_diff", "mean"),
    n=("pearsonr_nc_diff", "count"),
).reset_index()

hyperparam_summary = hyperparam_summary.sort_values("pearsonr_nc_mean", ascending=False)
hyperparam_summary.insert(0, "rank", np.arange(1, len(hyperparam_summary) + 1))

hyperparam_table = hyperparam_summary.copy()
hyperparam_table["pearsonr_nc_mean"] = hyperparam_table["pearsonr_nc_mean"].map(lambda x: f"{x:.5f}")
hyperparam_table["pearsonr_nc_sem"] = hyperparam_table["pearsonr_nc_sem"].map(lambda x: f"{x:.5f}")
hyperparam_table["rsa_c_test_mean"] = hyperparam_table["rsa_c_test_mean"].map(lambda x: f"{x:.5f}")
hyperparam_table["cka_c_test_mean"] = hyperparam_table["cka_c_test_mean"].map(lambda x: f"{x:.5f}")

hyperparam_table


In [ ]:
lr_order = sorted(hyperparam_summary["lr_label"].unique(), key=lr_label_to_float)
heatmap_data = hyperparam_summary.pivot(index="epochs", columns="lr_label", values="pearsonr_nc_mean")
heatmap_data = heatmap_data.loc[sorted(heatmap_data.index), lr_order] * HEATMAP_VAL_MULTIPLIER

fig, ax = plt.subplots(figsize=(6.5, 4), dpi=300)
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="vlag",
    center=0,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": r"$100 \times \Delta$ Average Alignment"},
    ax=ax,
)
ax.set_title("ViT-S Hyperparameter Sweep", fontsize=14, fontweight="bold")
ax.set_xlabel("Learning Rate", fontweight="bold")
ax.set_ylabel("Epochs", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout()

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="finetuning-hyperopt-hyperparams-average_heatmap",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Hyperparameter Sweep**

\textbf{finetuning-hyperopt-hyperparams-average_heatmap:} ViT-S hyperparameter sweep showing benchmark-average $100 \times \Delta r_{\mathrm{NC}}$ for each epoch and learning-rate setting, averaged across fine-tuning datasets and seeds.


In [ ]:
hyperparam_by_finetuning_dataset = hyperparam_df.groupby(
    ["finetuning_dataset", "finetuning_dataset_label", "epochs", "lr_label"],
    dropna=False,
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
).reset_index()

lr_order = sorted(hyperparam_by_finetuning_dataset["lr_label"].dropna().unique(), key=lr_label_to_float)
dataset_heatmap_values = hyperparam_by_finetuning_dataset["pearsonr_nc_mean"] * HEATMAP_VAL_MULTIPLIER
heatmap_absmax = np.nanmax(np.abs(dataset_heatmap_values))

fig, axes = plt.subplots(2, 3, figsize=(11.5, 5.8), dpi=300)
axes = axes.flatten()

for idx, (ax, dataset) in enumerate(zip(axes, FINETUNING_DATASETS)):
    data = hyperparam_by_finetuning_dataset[
        hyperparam_by_finetuning_dataset["finetuning_dataset"] == dataset
    ].copy()
    heatmap_data = data.pivot_table(
        index="epochs",
        columns="lr_label",
        values="pearsonr_nc_mean",
        aggfunc="mean",
    ).loc[sorted(data["epochs"].unique()), lr_order] * HEATMAP_VAL_MULTIPLIER

    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        vmin=-heatmap_absmax,
        vmax=heatmap_absmax,
        linewidths=0.5,
        linecolor="white",
        cbar=False,
        ax=ax,
    )
    ax.set_title(BENCHMARK_NAME_MAPPING[dataset], fontsize=14, fontweight="bold")
    ax.set_xlabel("Learning Rate", fontweight="bold")
    ax.set_ylabel("Epochs" if idx % 3 == 0 else "", fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle("ViT-S Hyperparameter Sweep by Fine-Tuning Dataset", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 1])
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(axes[-1].collections[0], cax=cbar_ax, label=r"$100 \times \Delta$ Average Alignment")

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="finetuning-hyperopt-hyperparams-by_finetuning_dataset_heatmap",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Hyperparameter Sweep by Fine-Tuning Dataset**

\textbf{finetuning-hyperopt-hyperparams-by_finetuning_dataset_heatmap:} Benchmark-average $100 \times \Delta r_{\mathrm{NC}}$ in the ViT-S sweep, shown separately for each fine-tuning dataset and averaged across seeds.


In [ ]:
hyperparam_by_benchmark = df_diff[
    df_diff["model_name"] == "ViT-S"
].groupby(
    ["benchmark_name", "benchmark_label", "epochs", "lr_label"],
    dropna=False,
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
).reset_index()

lr_order = sorted(hyperparam_by_benchmark["lr_label"].dropna().unique(), key=lr_label_to_float)
benchmark_heatmap_values = hyperparam_by_benchmark["pearsonr_nc_mean"] * HEATMAP_VAL_MULTIPLIER
heatmap_absmax = np.nanmax(np.abs(benchmark_heatmap_values))

fig, axes = plt.subplots(3, 3, figsize=(11.5, 8.5), dpi=300)
axes = axes.flatten()

for idx, (ax, benchmark) in enumerate(zip(axes, BENCHMARKS_WITH_AVERAGE)):
    data = hyperparam_by_benchmark[
        hyperparam_by_benchmark["benchmark_name"] == benchmark
    ].copy()
    heatmap_data = data.pivot_table(
        index="epochs",
        columns="lr_label",
        values="pearsonr_nc_mean",
        aggfunc="mean",
    ).loc[sorted(data["epochs"].unique()), lr_order] * HEATMAP_VAL_MULTIPLIER

    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        vmin=-heatmap_absmax,
        vmax=heatmap_absmax,
        linewidths=0.5,
        linecolor="white",
        cbar=False,
        ax=ax,
    )
    ax.set_title(BENCHMARK_NAME_MAPPING[benchmark], fontsize=14, fontweight="bold")
    ax.set_xlabel("Learning Rate", fontweight="bold")
    ax.set_ylabel("Epochs" if idx % 3 == 0 else "", fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle("ViT-S Hyperparameter Sweep by Evaluation Benchmark", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 1])
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(axes[-1].collections[0], cax=cbar_ax, label=r"$100 \times \Delta$ Alignment")

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="finetuning-hyperopt-hyperparams-by_benchmark_heatmap",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Hyperparameter Sweep by Benchmark**

\textbf{finetuning-hyperopt-hyperparams-by_benchmark_heatmap:} ViT-S hyperparameter effects, $100 \times \Delta r_{\mathrm{NC}}$, separated by evaluation benchmark and averaged across fine-tuning datasets and seeds.


In [ ]:
hyperparam_same_dataset = df_diff[
    (df_diff["model_name"] == "ViT-S")
    & (df_diff["finetuning_dataset"] == df_diff["benchmark_name"])
].groupby(
    ["finetuning_dataset", "finetuning_dataset_label", "epochs", "lr_label"],
    dropna=False,
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
).reset_index()

lr_order = sorted(hyperparam_same_dataset["lr_label"].dropna().unique(), key=lr_label_to_float)
same_dataset_heatmap_values = hyperparam_same_dataset["pearsonr_nc_mean"] * HEATMAP_VAL_MULTIPLIER
heatmap_absmax = np.nanmax(np.abs(same_dataset_heatmap_values))

fig, axes = plt.subplots(2, 3, figsize=(11.5, 5.8), dpi=300)
axes = axes.flatten()

for idx, (ax, dataset) in enumerate(zip(axes, FINETUNING_DATASETS)):
    data = hyperparam_same_dataset[
        hyperparam_same_dataset["finetuning_dataset"] == dataset
    ].copy()
    heatmap_data = data.pivot_table(
        index="epochs",
        columns="lr_label",
        values="pearsonr_nc_mean",
        aggfunc="mean",
    ).loc[sorted(data["epochs"].unique()), lr_order] * HEATMAP_VAL_MULTIPLIER

    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        vmin=-heatmap_absmax,
        vmax=heatmap_absmax,
        linewidths=0.5,
        linecolor="white",
        cbar=False,
        ax=ax,
    )
    ax.set_title(BENCHMARK_NAME_MAPPING[dataset], fontsize=14, fontweight="bold")
    ax.set_xlabel("Learning Rate", fontweight="bold")
    ax.set_ylabel("Epochs" if idx % 3 == 0 else "", fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle("ViT-S Hyperparameter Sweep: Same Fine-Tuning and Evaluation Dataset", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 1])
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(axes[-1].collections[0], cax=cbar_ax, label=r"$100 \times \Delta$ Alignment")

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="finetuning-hyperopt-hyperparams-same_dataset_heatmap",
    dpi=300,
    formats=["pdf", "png", "svg"],
)


**Caption: Hyperparameter Sweep for Matching Fine-Tuning and Evaluation Datasets**

\textbf{finetuning-hyperopt-hyperparams-same_dataset_heatmap:} ViT-S hyperparameter effects, $100 \times \Delta r_{\mathrm{NC}}$, when the fine-tuning and evaluation dataset match, averaged across seeds.


In [ ]:
MODALITY_MAP = {
    "bs_fz": "EP",
    "bs_mh": "EP",
    "tvsd": "EP",
    "things_fmri": "fMRI",
    "nsd_func1pt8mm_individualROIs": "fMRI",
    "things_eeg1": "EEG/MEG",
    "things_eeg2": "EEG/MEG",
    "things_meg": "EEG/MEG",
}

hyperparam_transfer = df_diff[
    (df_diff["model_name"] == "ViT-S")
    & (df_diff["benchmark_name"] != "benchmark_average")
].copy()
hyperparam_transfer["benchmark_modality"] = hyperparam_transfer["benchmark_name"].map(MODALITY_MAP)
hyperparam_transfer["finetuning_modality"] = hyperparam_transfer["finetuning_dataset"].map(MODALITY_MAP)
hyperparam_transfer["transfer_type"] = np.select(
    [
        hyperparam_transfer["benchmark_name"] == hyperparam_transfer["finetuning_dataset"],
        hyperparam_transfer["benchmark_modality"] == hyperparam_transfer["finetuning_modality"],
    ],
    ["Same dataset", "Same modality"],
    default="Cross modality",
)

transfer_type_order = ["Same dataset", "Same modality", "Cross modality"]
transfer_summary = hyperparam_transfer.groupby(
    ["transfer_type", "epochs", "lr_label"],
    dropna=False,
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
    pearsonr_nc_sem=("pearsonr_nc_diff", "sem"),
    n=("pearsonr_nc_diff", "count"),
).reset_index()

transfer_absmax = np.nanmax(np.abs(transfer_summary["pearsonr_nc_mean"] * HEATMAP_VAL_MULTIPLIER))

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6), dpi=300, sharey=True)

for idx, (ax, transfer_type) in enumerate(zip(axes, transfer_type_order)):
    data = transfer_summary[transfer_summary["transfer_type"] == transfer_type].copy()
    heatmap_data = data.pivot_table(
        index="epochs",
        columns="lr_label",
        values="pearsonr_nc_mean",
        aggfunc="mean",
    ).loc[sorted(data["epochs"].unique()), lr_order] * HEATMAP_VAL_MULTIPLIER

    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0,
        vmin=-transfer_absmax,
        vmax=transfer_absmax,
        linewidths=0.5,
        linecolor="white",
        cbar=False,
        ax=ax,
    )
    ax.set_title(transfer_type, fontsize=14, fontweight="bold")
    ax.set_xlabel("Learning Rate", fontweight="bold")
    ax.set_ylabel("Epochs" if idx == 0 else "", fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

fig.suptitle("Hyperparameter Effects by Transfer Type", fontsize=16, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 1])
cbar_ax = fig.add_axes([0.92, 0.2, 0.015, 0.65])
fig.colorbar(axes[-1].collections[0], cax=cbar_ax, label=r"$100 \times \Delta$ Alignment")

save_figs(
    fig=fig,
    save_dir=save_dir,
    base_filename="finetuning-hyperopt-hyperparams-transfer_type_heatmap",
    dpi=300,
    formats=["pdf", "png", "svg"],
)

transfer_summary.sort_values(["transfer_type", "epochs", "lr_label"])


**Caption: Hyperparameter Transfer-Type Diagnostic**

\textbf{finetuning-hyperopt-hyperparams-transfer_type_heatmap:} ViT-S hyperparameter effects, $100 \times \Delta r_{\mathrm{NC}}$, grouped into same-dataset, same-modality, and cross-modality transfer comparisons.


## Overall Analysis

\paragraph{Model size.} We first test whether the neural-alignment gain from fine-tuning varies with ViT backbone size at the common setting of $20$ epochs and learning rate $10^{-5}$. Figure~\ref{fig:finetuning-hyperopt-model_size-by_benchmark_dataset} resolves each combination of fine-tuning dataset and evaluation benchmark, and Figure~\ref{fig:finetuning-hyperopt-model_size-average_by_benchmark} averages over fine-tuning datasets. The benchmark-average gain is largest for ViT-L ($\Delta r_{\mathrm{NC}}=0.00737 \pm 0.00050$ SEM), followed by ViT-S ($0.00591 \pm 0.00073$) and ViT-B ($0.00366 \pm 0.00047$); model size therefore does not yield a monotonic improvement. Capacity helps most clearly for T-fMRI, where mean gains increase from ViT-S to ViT-B to ViT-L ($0.00324$, $0.01477$, and $0.01911$). Conversely, MH-EP and T-MEG favor ViT-S on average ($0.01118$ and $0.00794$, respectively), showing that the benefit of the larger model depends on the evaluation benchmark.

\paragraph{Hyperparameter sweep.} Figure~\ref{fig:finetuning-hyperopt-hyperparams-average_heatmap} provides evidence for the $20$-epoch, $10^{-5}$ learning-rate setting used in the fine-tuning experiments. This setting yields a benchmark-average gain of $\Delta r_{\mathrm{NC}}=0.00591 \pm 0.00073$ SEM, which is effectively identical to the largest observed gain at $50$ epochs and $5\times10^{-6}$ ($0.00594 \pm 0.00086$; difference $=0.00003$), while requiring less than half as many training epochs. It is also comparable to $100$ epochs at $10^{-6}$ ($0.00581 \pm 0.00070$). Thus, $20$ epochs at $10^{-5}$ provides a near-maximal alignment gain with the shortest training schedule among the top-performing settings. In contrast, stronger updates substantially reduce alignment: $100$ epochs at $5\times10^{-4}$ produces $\Delta r_{\mathrm{NC}}=-0.05775 \pm 0.00823$. Figure~\ref{fig:finetuning-hyperopt-hyperparams-by_finetuning_dataset_heatmap} shows that the precise best low-rate setting differs by fine-tuning source, but the drop at larger learning rates recurs across datasets. Figure~\ref{fig:finetuning-hyperopt-hyperparams-by_benchmark_heatmap} further localizes this drop: T-fMRI, T-EEG2, and T-MEG decrease most strongly as learning rate and training duration increase, whereas FZ-EP remains comparatively tolerant of larger learning rates.

\paragraph{Matching and transfer evaluations.} Figure~\ref{fig:finetuning-hyperopt-hyperparams-same_dataset_heatmap} shows that aggressive updates also harm alignment when models are evaluated on the dataset used for fine-tuning, especially for T-fMRI, T-EEG2, and T-MEG. Figure~\ref{fig:finetuning-hyperopt-hyperparams-transfer_type_heatmap} summarizes this effect by transfer relationship. At the selected setting of $20$ epochs and learning rate $10^{-5}$, gains are positive for same-dataset ($0.0112$), same-modality ($0.0058$), and cross-modality ($0.0049$) comparisons. At $100$ epochs and learning rate $5\times10^{-4}$, all three become negative, with the largest decrease in the same-dataset comparison ($-0.1615$, compared with $-0.0511$ for same-modality and $-0.0404$ for cross-modality comparisons). Thus, the selected configuration combines near-maximal average performance with positive transfer across all comparison types, whereas stronger updating generally degrades alignment. Overall, the model-size analysis indicates benchmark-dependent capacity effects, and the hyperparameter sweep supports the $20$-epoch, $10^{-5}$ setting used for the fine-tuning experiments.



In [ ]:
best_by_dataset = hyperparam_df.groupby(
    ["finetuning_dataset", "finetuning_dataset_label", "epochs", "lr_label", "hyperparam"],
    dropna=False,
).agg(
    pearsonr_nc_mean=("pearsonr_nc_diff", "mean"),
    pearsonr_nc_sem=("pearsonr_nc_diff", "sem"),
    n=("pearsonr_nc_diff", "count"),
).reset_index()

best_by_dataset = best_by_dataset.sort_values(
    ["finetuning_dataset", "pearsonr_nc_mean"], ascending=[True, False]
)
best_by_dataset.groupby("finetuning_dataset_label", sort=False).head(3)
